# WorkflowContext from raw data

Ce notebook initialise un `WorkflowContext` a partir d'un dossier de fichiers bruts, puis fournit quelques cellules utiles pour inspecter `runs`, `configurations`, `issues` et generer les fichiers `refs_sub` / `refs_norm`.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")
print(f"Python path includes: {SRC}")

Repository root: /home/achennev/python/scarlet
Python path includes: /home/achennev/python/scarlet/src


In [2]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory

RAW_DIR = ROOT / "data" / "SANSLLB" / "raw"
OUTPUT_DIR = ROOT / "data" / "SANSLLB" / "out"
INSTRUMENT_NAME = "sansllb"

RAW_DIR, OUTPUT_DIR

(PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw'),
 PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out'))

In [3]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    output_dir=OUTPUT_DIR,
    overwrite=True
)

print(f"runs: {len(w.runs)}")
print(f"configurations: {len(w.configurations)}")
print(f"issues: {len(w.issues)}")
print(f"artifacts: {len(w.artifacts)}")

runs: 62
configurations: 2
issues: 4
artifacts: 64


In [4]:
w.runs_table()

sample_name,config_id,mode,entity,file_path
empty_beam_att5_ws_beamstop,config_1,transmission,empty_beam,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs
empty_cell,config_1,transmission,empty_cell,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs
H2O,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002342.nxs
S1_P_PB_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002343.nxs
S2_P_PB_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002344.nxs
S3_P_PBK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002345.nxs
S4_P_BC_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002346.nxs
S5_P_BC_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002347.nxs
S6_P_BCK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002348.nxs
S7_P_MI_25_1mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002349.nxs


In [5]:
w.configurations_table()

config_id,wavelength,sample_detector_distance,notes,has_collimation,collimation_distance,last_aperture_to_sample_distance,aperture1_type,aperture1_x_gap,aperture1_y_gap,aperture1_diameter,aperture2_type,aperture2_x_gap,aperture2_y_gap,aperture2_diameter
config_1,6.00046,2.5,,True,0.5,1.2,slit,0.0419996,0.0419996,,slit,0.0319997,0.0320003,
config_2,5.9989,9,,True,2.58333,1.2,slit,0.0269997,0.0269999,,slit,0.0169996,0.0169998,


In [6]:
from scarlet.workflow.context import generate_reference_files_from_workflow_context

generate_reference_files_from_workflow_context(w)
print("refs_sub:", {k: str(v) for k, v in sorted(w.refs_sub_files.items())})
print("refs_norm:", {k: str(v) for k, v in sorted(w.refs_norm_files.items())})

refs_sub: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_2.nxs'}
refs_norm: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs'}


In [7]:
from scarlet.workflow import update_detector0_beam_center_from_empty_beam_transmission
for key in w.refs_sub_files:
    update_detector0_beam_center_from_empty_beam_transmission(w.refs_sub_files[key])

In [8]:
from scarlet.gui import run_mask_editor

# run_mask_editor()

In [9]:
# from scarlet.workflow.reference import insert_masks_in_refs_file
# for key in w.refs_norm_files:
#     insert_masks_in_refs_file(w.refs_norm_files[key], w.masks)
# for key in w.refs_sub_files:
#     insert_masks_in_refs_file(w.refs_sub_files[key], w.masks)   

In [10]:
from scarlet.workflow.context import write_runs_report_csv

In [11]:
file = write_runs_report_csv(w, overwrite=True)
print(f"Runs report written to: {file}")

Runs report written to: /home/achennev/python/scarlet/data/SANSLLB/out/runs_report.csv


In [12]:
# test change with respect to modified run csv file
# from scarlet.workflow.context import update_workflow_context_from_runs_report_csv
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report_modified.csv")
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report.csv")
# file = write_runs_report_csv(w, csv_path="data/SANSLLB/out/runs_report_updated.csv", overwrite=True)

In [13]:
from scarlet.reduction.transmission import compute_reference_transmissions
for key in w.refs_norm_files:
    compute_reference_transmissions(w.refs_norm_files[key])
for key in w.refs_sub_files:
    compute_reference_transmissions(w.refs_sub_files[key])


In [14]:
w.refs_norm_files

{'config_1': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs'),
 'config_2': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs')}

In [15]:
from scarlet.workflow import update_reference_masks_from_workflow_context

w = update_reference_masks_from_workflow_context(w)

In [16]:
from scarlet.workflow import write_corrected_water_scattering

for config_id, refs_norm_path in sorted(w.refs_norm_files.items()):
    write_corrected_water_scattering(refs_norm_path)
    print(config_id, refs_norm_path)

config_1 /home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs
config_2 /home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs


In [17]:
# Save workflow context to file
from scarlet.workflow import save_workflow_context, load_workflow_context

save_workflow_context(w, "workflow_context.nxs")
w2 = load_workflow_context("workflow_context.nxs")

In [18]:
from scarlet.workflow.context import update_transmissions_from_workflow_context
update_transmissions_from_workflow_context(w)
w.transmissions_table()

sample_name,config_id,transmission
H2O,config_1,0.545957
H2O,config_2,0.592539
S10_P_BB_60_1mm,config_1,0.874308
S10_P_BB_60_1mm,config_2,0.986359
S11_P_EB_25_1mm,config_1,0.885527
S11_P_EB_25_1mm,config_2,0.998175
S12_P_EB_60_1mm,config_1,0.878189
S12_P_EB_60_1mm,config_2,0.983866
S1_P_PB_25_2mm,config_1,0.826591
S1_P_PB_25_2mm,config_2,0.925727


In [19]:
from scarlet.workflow import integrate_scattering_from_workflow_context

integrate_scattering_from_workflow_context(w, n_bins=200)

/home/achennev/python/scarlet/src/scarlet/workflow/context.py:2084: RuntimeWarning: divide by zero encountered in divide
  solid_angle_corrected = solid_angle_corrected / water_corrected_image
/home/achennev/python/scarlet/src/scarlet/workflow/context.py:2084: RuntimeWarning: invalid value encountered in divide
  solid_angle_corrected = solid_angle_corrected / water_corrected_image
/home/achennev/python/scarlet/src/scarlet/workflow/context.py:2085: RuntimeWarning: divide by zero encountered in divide
  solid_angle_error = solid_angle_error / np.abs(water_corrected_image)
/home/achennev/python/scarlet/src/scarlet/workflow/context.py:2085: RuntimeWarning: invalid value encountered in divide
  solid_angle_error = solid_angle_error / np.abs(water_corrected_image)


WorkflowContext(experiment_id='experiment', instrument_name='sansllb', root_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw'), output_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out'), schema_raw='scarlet_nxsas_raw_v1.3_mono.yaml', schema_refs_sub='scarlet_refs_sub_v1.0.yaml', schema_refs_norm='scarlet_refs_norm_v1.0.yaml', schema_masks='scarlet_masks_v1.0.yaml', runs={RunKey(config_id='config_1', entity='empty_beam', mode='transmission', sample_name='empty_beam_att5_ws_beamstop'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs'), RunKey(config_id='config_1', entity='dark', mode='scattering', sample_name='Cd'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002356.nxs'), RunKey(config_id='config_1', entity='empty_cell', mode='transmission', sample_name='empty_cell'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs'), RunKey(config_id='config_1', entity='sample', mode

{'config_1': Configuration(wavelength=6.000461656749349, sample_detector_distance=2.5000035990000002, collimation=Collimation(aperture1=Aperture(type='slit', x_gap=0.041999600000000005, y_gap=0.041999600000000005, diameter=None), aperture2=Aperture(type='slit', x_gap=0.0319997, y_gap=0.032000299999999995, diameter=None), collimation_distance=0.5000000000000002, last_aperture_to_sample_distance=1.2), config_id='config_1', notes=None),
 'config_2': Configuration(wavelength=5.998901477173986, sample_detector_distance=8.999996799, collimation=Collimation(aperture1=Aperture(type='slit', x_gap=0.026999699999999998, y_gap=0.0269999, diameter=None), aperture2=Aperture(type='slit', x_gap=0.0169996, y_gap=0.016999800000000002, diameter=None), collimation_distance=2.5833333333333313, last_aperture_to_sample_distance=1.2), config_id='config_2', notes=None)}